# LUAD preprocessing

Prepare the canonical scanVI AnnData used by the public LUAD workflow. The linked source dataset is read-only; all generated files are written under this experiment directory.


In [ ]:
from pathlib import Path
import sys
import yaml

DATASET = "LUAD"
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
sys.path.append(str(REPO_ROOT / "src"))

with (EXPERIMENT_DIR / "config.yaml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def experiment_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
SOURCE_ADATA = DATA_ROOT / "source.h5ad"
SCANVI_DIR.mkdir(parents=True, exist_ok=True)


## Load and audit the read-only source


In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
import scvi

if not SOURCE_ADATA.is_file():
    raise FileNotFoundError(f"Missing linked source AnnData: {SOURCE_ADATA}")

adata = ad.read_h5ad(SOURCE_ADATA)
required_obs = {"status", "_clone_s"}
missing_obs = sorted(required_obs - set(adata.obs.columns))
if missing_obs:
    raise KeyError(f"Source AnnData is missing obs fields: {missing_obs}")
if "counts" not in adata.layers:
    raise KeyError("Source AnnData is missing adata.layers['counts']")
if "spatial" not in adata.obsm:
    raise KeyError("Source AnnData is missing adata.obsm['spatial']")

adata.obs["status"] = adata.obs["status"].astype(str).astype("category")
adata.obs["_clone_s"] = adata.obs["_clone_s"].astype(str).astype("category")
adata


## Train scanVI without time input

The saved AnnData is the single model-ready input for Stage 1, Stage 2, and decoder training.


In [ ]:
scvi.model.SCANVI.setup_anndata(
    adata,
    layer="counts",
    labels_key="_clone_s",
    unlabeled_category="Unknown",
    batch_key="status",
)
model = scvi.model.SCANVI(adata)
model.train()


## Align AAH coordinates to the LUAD reference

Reproduce the coordinate processing from the original LUAD workflow using a joint scanVI UMAP representation and feature-guided rigid UOT alignment.


In [ ]:
from stvirtual.models import uot_alignment_trans as alignment

LATENT_KEY = CONFIG.get("latent_key", "X_scanVI")
adata.obsm[LATENT_KEY] = model.get_latent_representation()
sc.pp.neighbors(adata, use_rep=LATENT_KEY)
sc.tl.umap(adata)

source_mask = adata.obs["status"].astype(str).to_numpy() == "AAH"
reference_mask = adata.obs["status"].astype(str).to_numpy() == "LUAD"
if not source_mask.any() or not reference_mask.any():
    raise ValueError("Both AAH and LUAD observations are required for coordinate alignment")

spatial = np.asarray(adata.obsm["spatial"][:, :2], dtype=np.float64)
X_src = spatial[source_mask]
X_ref = spatial[reference_mask]
spatial_all = np.vstack([X_src, X_ref])
spatial_mean = spatial_all.mean(axis=0, keepdims=True)
spatial_std = spatial_all.std(axis=0, keepdims=True) + 1e-8
X_src_normalized = (X_src - spatial_mean) / spatial_std
X_ref_normalized = (X_ref - spatial_mean) / spatial_std

features = np.asarray(adata.obsm["X_umap"], dtype=np.float64)
F_src = features[source_mask]
F_ref = features[reference_mask]
feature_all = np.vstack([F_src, F_ref])
feature_min = feature_all.min(axis=0, keepdims=True)
feature_range = np.maximum(feature_all.max(axis=0, keepdims=True) - feature_min, 1e-8)
F_src = (F_src - feature_min) / feature_range
F_ref = (F_ref - feature_min) / feature_range

labels = adata.obs["_clone_s"].astype(str).to_numpy()
X_src_aligned_normalized, alignment_info = alignment.register_slice_to_ref_by_features(
    X_src_normalized,
    X_ref_normalized,
    F_src,
    F_ref,
    ann_src=labels[source_mask],
    ann_ref=labels[reference_mask],
    k_feat=32,
    match_mode="soft",
    alpha_xy=1.0,
    beta_feat=3.0,
    tau=0.8,
    seed=2025,
    verbose=True,
)
X_src_aligned = X_src_aligned_normalized * spatial_std + spatial_mean

spatial_aligned = spatial.copy()
spatial_aligned[source_mask] = X_src_aligned
adata.obs["cx_aligned"] = spatial_aligned[:, 0]
adata.obs["cy_aligned"] = spatial_aligned[:, 1]
adata.obsm["spatial_aligned"] = spatial_aligned.astype(np.float32)

print(f"Aligned {source_mask.sum()} AAH spots to {reference_mask.sum()} LUAD reference spots")


## Save the canonical model and AnnData


In [ ]:
model.save(SCANVI_DIR, overwrite=True, save_anndata=True)
adata.write_h5ad(SCANVI_ADATA, compression="gzip")
print(f"Saved model-ready AnnData: {SCANVI_ADATA}")


## Validate the public contract


In [ ]:
check = ad.read_h5ad(SCANVI_ADATA, backed="r")
try:
    summary = {
        "shape": check.shape,
        "statuses": check.obs["status"].astype(str).value_counts().to_dict(),
        "latent_shape": tuple(check.obsm[LATENT_KEY].shape),
        "counts_shape": tuple(check.layers["counts"].shape),
        "aligned_coordinates": all(key in check.obs for key in ("cx_aligned", "cy_aligned")),
    }
finally:
    check.file.close()
summary
